# A donde va el turista cuando desaparece la VUT

Las **tres opciones** montadas en el mismo cuaderno, para verlas y decidir cual va a la web.

| | Que hace | Que asume |
|---|---|---|
| **1. Peso manual** | Un control 0-100% entre precio y cercania; el reparto se recalcula | Nada. La decision es de quien mira |
| **2. Regresion** | Estima el peso a partir de la demanda real de Airbnb hoy | Que el turista de 2028 decide como el de hoy |
| **3. Las dos** | La regresion pone el valor por defecto, el control lo mueve | Lo mismo, pero se puede discutir |

## Lo que NO se puede hacer

**No hay regresion posible de "a donde va el turista en 2028": 2028 no ha pasado y no hay
variable que predecir.** Lo que si hay es la demanda de hoy --`number_of_reviews_ltm`, resenas del
ultimo ano, en los 6.834 anuncios y sin nulos-- que permite medir cuanto pesan precio y
localizacion **ahora**. Los pesos de hoy se aplican al reparto de 2028; eso es un supuesto, no un
resultado.

Los hoteles no tienen ninguna columna de demanda, asi que la regresion solo se puede ajustar sobre
Airbnb.

In [1]:
# 1. Carga
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
GOLD = RAIZ / "data" / "gold"

vut = pd.read_csv(GOLD / "airbnb_para_web.csv", low_memory=False)
hoteles = pd.read_csv(GOLD / "alojamientos_reglados.csv", low_memory=False)
hoteles = hoteles[hoteles["banda_plaza"].notna() & hoteles["lat"].notna()].copy()
restauracion = pd.read_csv(GOLD / "restauracion_bcn.csv", low_memory=False)

print(f"VUT en Airbnb        {len(vut):>7,}  plazas {int(vut['accommodates'].sum()):>7,}")
print(f"Alojamiento reglado  {len(hoteles):>7,}  plazas {int(hoteles['plazas'].sum()):>7,}")
print(f"Restauracion         {len(restauracion):>7,}")

VUT en Airbnb          6,834  plazas  30,067
Alojamiento reglado      762  plazas  84,314
Restauracion           9,479


## 1. Variables comunes

Las dos ofertas tienen que medirse igual para poder compararse: precio por plaza y noche
(equivalente anual) y distancia al centro en kilometros.

In [2]:
# 2. Distancia al centro y precio por plaza
CENTRO = (41.3870, 2.1700)   # Placa de Catalunya


def distancia_km(lat, lon, centro=CENTRO):
    """Haversine. Con geometria plana el eje este-oeste sale inflado un tercio a esta latitud."""
    lat1, lon1 = np.radians(centro)
    lat2, lon2 = np.radians(np.asarray(lat, float)), np.radians(np.asarray(lon, float))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


vut["dist_centro"] = distancia_km(vut["latitude"], vut["longitude"])
vut["precio"] = vut["precio_plaza_anual"]
hoteles["dist_centro"] = distancia_km(hoteles["lat"], hoteles["lon"])
hoteles["precio"] = hoteles["precio_plaza"]

resumen = pd.DataFrame({
    "VUT": [vut["precio"].median(), vut["dist_centro"].median(), vut["accommodates"].sum()],
    "Hoteles": [hoteles["precio"].median(), hoteles["dist_centro"].median(),
                hoteles["plazas"].sum()],
}, index=["precio mediano EUR/plaza", "distancia mediana km", "plazas totales"]).round(1)
display(resumen)

print(f"Plazas VUT a reubicar: {int(vut['accommodates'].sum()):,}")
print(f"Plazas reglado        : {int(hoteles['plazas'].sum()):,}")
print(f"La oferta reglada es {hoteles['plazas'].sum() / vut['accommodates'].sum():.1f} veces la VUT")

,VUT,Hoteles
precio mediano EUR/plaza,51.8,77.8
distancia mediana km,1.7,1.2
plazas totales,30067.0,84314.0


Plazas VUT a reubicar: 30,067
Plazas reglado        : 84,314
La oferta reglada es 2.8 veces la VUT


# OPCION 2 --- La regresion

Se estima primero porque de ella sale el valor por defecto de la opcion 3.

**Variable dependiente:** `number_of_reviews_ltm`, resenas del ultimo ano. Es un proxy de demanda:
mide reservas realizadas, no preferencia declarada, y solo cuenta a quien deja resena. Como
alrededor de un tercio no resena, el nivel esta subestimado, pero el **orden** entre anuncios se
mantiene, que es lo que importa aqui.

In [3]:
# 3. La regresion de demanda
import statsmodels.api as sm

d = vut[(vut["number_of_reviews_ltm"] > 0) & (vut["precio"] > 0)].copy()

# Logaritmos en demanda y precio: asi los coeficientes son elasticidades --cuanto cambia la demanda
# en % cuando el precio sube un 1%-- y no dependen de las unidades.
d["log_demanda"] = np.log(d["number_of_reviews_ltm"])
d["log_precio"] = np.log(d["precio"])

X = d[["log_precio", "dist_centro", "accommodates"]].copy()
X["entera"] = (d["room_type"] == "Entire home/apt").astype(int)
X = sm.add_constant(X)
modelo = sm.OLS(d["log_demanda"], X).fit()

print(f"n = {len(d):,}   R2 = {modelo.rsquared:.3f}")
print()
tabla = pd.DataFrame({
    "coef": modelo.params.round(4),
    "p": modelo.pvalues.round(4),
    "IC bajo": modelo.conf_int()[0].round(3),
    "IC alto": modelo.conf_int()[1].round(3),
})
display(tabla)

print("Lectura directa:")
print(f"  precio     +1%  -> demanda {modelo.params['log_precio']:+.2f}%")
print(f"  distancia  +1km -> demanda {(np.exp(modelo.params['dist_centro']) - 1) * 100:+.1f}%")

n = 6,834   R2 = 0.122



,coef,p,IC bajo,IC alto
const,-1.5663,0.0000,-1.854,-1.279
log_precio,0.7860,0.0000,0.722,0.850
dist_centro,0.0097,0.4827,-0.017,0.037
accommodates,0.0825,0.0000,0.068,0.097
entera,0.7591,0.0000,0.640,0.878


Lectura directa:
  precio     +1%  -> demanda +0.79%
  distancia  +1km -> demanda +1.0%


In [4]:
# 4. Cual pesa mas: coeficientes estandarizados
#
# Los coeficientes crudos no se pueden comparar: uno esta en % y otro en kilometros. Estandarizando
# --restar la media y dividir por la desviacion-- los dos quedan en "cuanto cambia la demanda
# cuando esta variable se mueve una desviacion tipica", que si es comparable.
Xz = X.drop(columns="const")
Xz = (Xz - Xz.mean()) / Xz.std()
mz = sm.OLS((d["log_demanda"] - d["log_demanda"].mean()) / d["log_demanda"].std(),
            sm.add_constant(Xz)).fit()

betas = mz.params.drop("const").abs().sort_values(ascending=False)
print("=== PESO RELATIVO (beta estandarizada, en valor absoluto) ===")
for v, b in betas.items():
    print(f"  {v:<14} {b:.3f}  {'#' * int(b * 100)}")

peso_precio = betas["log_precio"] / (betas["log_precio"] + betas["dist_centro"])
print()
print(f"Reparto entre las dos que nos interesan:")
print(f"  PRECIO      {peso_precio:.0%}")
print(f"  DISTANCIA   {1 - peso_precio:.0%}")

=== PESO RELATIVO (beta estandarizada, en valor absoluto) ===
  log_precio     0.288  ############################
  entera         0.150  ###############
  accommodates   0.142  ##############
  dist_centro    0.008  

Reparto entre las dos que nos interesan:
  PRECIO      97%
  DISTANCIA   3%


In [5]:
# 5. Lo que invalida la lectura ingenua: precio y distancia van juntos
#
# El centro es caro PORQUE es el centro. Si las dos variables miden en parte lo mismo, repartir el
# peso entre ellas es una cuenta discutible, no un hecho.
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("=== CORRELACION ===")
print(d[["log_precio", "dist_centro", "accommodates"]].corr().round(3).to_string())
print()
print("=== VIF (factor de inflacion de la varianza) ===")
print("  por encima de 5 la variable comparte demasiada informacion con las otras")
base = sm.add_constant(d[["log_precio", "dist_centro", "accommodates"]])
for i, v in enumerate(base.columns):
    if v != "const":
        print(f"  {v:<14} {variance_inflation_factor(base.values, i):.2f}")
print()
# Cuanto explica cada una por separado, frente a las dos juntas
solo_p = sm.OLS(d["log_demanda"], sm.add_constant(d[["log_precio"]])).fit().rsquared
solo_d = sm.OLS(d["log_demanda"], sm.add_constant(d[["dist_centro"]])).fit().rsquared
juntas = sm.OLS(d["log_demanda"], sm.add_constant(d[["log_precio", "dist_centro"]])).fit().rsquared
print(f"R2 solo precio    {solo_p:.3f}")
print(f"R2 solo distancia {solo_d:.3f}")
print(f"R2 las dos        {juntas:.3f}   (suma por separado: {solo_p + solo_d:.3f})")
print()
if juntas < solo_p + solo_d:
    print("Las dos juntas explican MENOS que la suma por separado: se solapan.")

=== CORRELACION ===
              log_precio  dist_centro  accommodates
log_precio         1.000       -0.131        -0.245
dist_centro       -0.131        1.000        -0.070
accommodates      -0.245       -0.070         1.000

=== VIF (factor de inflacion de la varianza) ===
  por encima de 5 la variable comparte demasiada informacion con las otras
  log_precio     1.09
  dist_centro    1.03
  accommodates   1.08

R2 solo precio    0.068
R2 solo distancia 0.001
R2 las dos        0.068   (suma por separado: 0.069)

Las dos juntas explican MENOS que la suma por separado: se solapan.


# OPCION 1 --- El peso manual

Cada VUT que desaparece busca alojamiento reglado. Se puntua cada hotel con:

    utilidad = w x cercania + (1 - w) x parecido_de_precio

`w = 1` es un turista que solo quiere quedarse en el mismo sitio. `w = 0`, uno que solo quiere
pagar lo mismo. **Nada de esto se estima: `w` lo pone quien mira.**

In [6]:
# 6. El modelo de sustitucion
from scipy.spatial import cKDTree

# Distancias VUT -> hotel en km, con la misma proyeccion local de siempre.
LAT0 = 41.39


def proyectar(lat, lon):
    return np.c_[np.asarray(lon, float) * 111.320 * np.cos(np.radians(LAT0)),
                 np.asarray(lat, float) * 110.570]


P_vut = proyectar(vut["latitude"], vut["longitude"])
P_hot = proyectar(hoteles["lat"], hoteles["lon"])

# 6.834 x 762 son 5,2 millones de pares: cabe en memoria de sobra.
D = np.sqrt(((P_vut[:, None, :] - P_hot[None, :, :]) ** 2).sum(axis=2))
BRECHA = np.abs(vut["precio"].to_numpy()[:, None] - hoteles["precio"].to_numpy()[None, :])

# Las dos magnitudes se llevan a 0-1 para poder mezclarlas: 1 es el mejor hotel posible.
cercania = 1 - D / D.max()
parecido = 1 - BRECHA / BRECHA.max()


def repartir(w: float) -> pd.DataFrame:
    """Cada VUT se lleva sus plazas al hotel de mayor utilidad. `w` es el peso de la cercania."""
    destino = (w * cercania + (1 - w) * parecido).argmax(axis=1)
    r = hoteles.iloc[destino].copy()
    r["plazas_recibidas"] = vut["accommodates"].to_numpy()
    r["km_recorridos"] = D[np.arange(len(vut)), destino]
    r["salto_precio"] = (hoteles["precio"].to_numpy()[destino] - vut["precio"].to_numpy())
    return r


print(f"matriz de distancias: {D.shape[0]:,} VUT x {D.shape[1]:,} hoteles")
print(f"distancia VUT-hotel mas cercano: mediana {np.median(D.min(axis=1)):.2f} km")

matriz de distancias: 6,834 VUT x 762 hoteles
distancia VUT-hotel mas cercano: mediana 0.10 km


In [7]:
# 7. Que cambia segun el peso
filas = []
for w in [0.0, 0.25, 0.5, 0.75, 1.0]:
    r = repartir(w)
    filas.append({
        "w (peso cercania)": w,
        "hoteles que reciben": r["licencia_id"].nunique(),
        "km medianos": round(float(r["km_recorridos"].median()), 2),
        "salto de precio mediano": round(float(r["salto_precio"].median()), 1),
        "plazas al hotel mas saturado": int(r.groupby("licencia_id")["plazas_recibidas"].sum().max()),
    })
display(pd.DataFrame(filas))

print("Con w=1 el turista no se mueve de barrio pero paga lo que sea.")
print("Con w=0 paga lo mismo pero puede acabar en la otra punta de la ciudad.")

,w (peso cercania),hoteles que reciben,km medianos,salto de precio mediano,plazas al hotel mas saturado
0,0.00,581,2.22,0.0,3038
1,0.25,634,0.33,1.1,1497
2,0.50,619,0.21,4.0,1256
3,0.75,618,0.14,9.4,808
4,1.00,650,0.10,25.6,526


Con w=1 el turista no se mueve de barrio pero paga lo que sea.
Con w=0 paga lo mismo pero puede acabar en la otra punta de la ciudad.


# OPCION 3 --- Las dos

El control sigue estando, pero arranca en el peso que dice la regresion en vez de en un 50%
arbitrario.

In [8]:
# 8. El reparto con el peso estimado
w_estimado = float(1 - peso_precio)   # la regresion da peso de PRECIO; aqui se usa el de CERCANIA
print(f"w por defecto segun la regresion: {w_estimado:.2f}")
print()

r = repartir(w_estimado)
por_barrio = (r.assign(plazas=r["plazas_recibidas"])
              .groupby("barrio")
              .agg(hoteles=("licencia_id", "nunique"), plazas_recibidas=("plazas", "sum"))
              .sort_values("plazas_recibidas", ascending=False))

print("=== BARRIOS QUE MAS PLAZAS ABSORBEN ===")
display(por_barrio.head(10))

capacidad = hoteles.groupby("barrio")["plazas"].sum()
por_barrio["capacidad"] = capacidad
por_barrio["ocupacion_extra_%"] = (por_barrio["plazas_recibidas"] / por_barrio["capacidad"] * 100).round(0)
print("=== DONDE NO CABEN: plazas recibidas frente a capacidad instalada ===")
display(por_barrio.sort_values("ocupacion_extra_%", ascending=False).head(10))

w por defecto segun la regresion: 0.03

=== BARRIOS QUE MAS PLAZAS ABSORBEN ===


,hoteles,plazas_recibidas
barrio,,
la Dreta de l'Eixample,163,10083.0
Sant Antoni,23,4257.0
el Raval,65,2804.0
la Vila de Gràcia,20,1411.0
l'Antiga Esquerra de l'Eixample,47,1290.0
el Barri Gòtic,84,1226.0
la Nova Esquerra de l'Eixample,20,820.0
la Sagrada Família,9,759.0
el Poble Sec,26,706.0


=== DONDE NO CABEN: plazas recibidas frente a capacidad instalada ===


,hoteles,plazas_recibidas,capacidad,ocupacion_extra_%
barrio,,,,
la Bordeta,2,244.0,39.0,626.0
Vilapicina i la Torre Llobeta,2,83.0,20.0,415.0
Sant Antoni,23,4257.0,1402.0,304.0
la Vila de Gràcia,20,1411.0,738.0,191.0
Sant Genís dels Agudells,1,26.0,14.0,186.0
el Guinardó,6,610.0,441.0,138.0
la Sagrada Família,9,759.0,599.0,127.0
el Baix Guinardó,5,419.0,369.0,114.0
el Camp d'en Grassot i Gràcia Nova,5,406.0,439.0,92.0


In [9]:
# 9. Las tres, una al lado de la otra
comparacion = []
for etiqueta, w in [("1. manual, w=0,5 (neutro)", 0.5),
                    (f"2. regresion, w={w_estimado:.2f}", w_estimado),
                    ("1. manual, w=1 (solo cercania)", 1.0),
                    ("1. manual, w=0 (solo precio)", 0.0)]:
    rr = repartir(w)
    comparacion.append({
        "opcion": etiqueta,
        "hoteles usados": rr["licencia_id"].nunique(),
        "km medianos": round(float(rr["km_recorridos"].median()), 2),
        "sobrecoste mediano EUR/plaza": round(float(rr["salto_precio"].median()), 1),
        "barrios con destino": rr["barrio"].nunique(),
    })
display(pd.DataFrame(comparacion).set_index("opcion"))

,hoteles usados,km medianos,sobrecoste mediano EUR/plaza,barrios con destino
opcion,,,,
"1. manual, w=0,5 (neutro)",619,0.21,4.0,48
"2. regresion, w=0.03",653,0.80,0.1,46
"1. manual, w=1 (solo cercania)",650,0.10,25.6,50
"1. manual, w=0 (solo precio)",581,2.22,0.0,50


# El reparto con capacidad

Todo lo anterior ignora que un hotel se llena. Un establecimiento de 200 plazas no puede absorber
las 500 de las VUT que tiene al lado.

**Como se reparte.** Las VUT se ordenan de mayor a menor precio y van escogiendo por turnos: la
mas cara elige primero. Cada una se lleva sus plazas al hotel de mayor utilidad **que aun tenga
sitio**, y si no cabe entera se parte entre varios.

**Por que las caras primero.** Es el supuesto que hace falta para que el reparto tenga un orden, y
no es neutral: quien paga mas acaba mas cerca y en mejor categoria. La alternativa --sorteo-- se
compara al final.

**La segmentacion por categoria no se impone: sale sola.** El termino de parecido de precio ya
manda a una VUT de 120 EUR/plaza hacia los cinco estrellas (mediana 166 EUR) y a una de 30 hacia
hostales y pensiones (mediana 66 EUR). La celda de comprobacion ensena si de verdad ocurre.

In [10]:
# 10. Reparto con capacidad y prioridad
def repartir_con_capacidad(w: float, prioridad: str = "precio") -> tuple:
    """Devuelve (asignaciones, hoteles con capacidad restante).

    `prioridad`: 'precio' -> las VUT caras eligen primero; 'sorteo' -> orden aleatorio.
    """
    utilidad = w * cercania + (1 - w) * parecido
    orden_hoteles = np.argsort(-utilidad, axis=1)          # mejor hotel primero, por VUT

    if prioridad == "precio":
        turno = np.argsort(-vut["precio"].to_numpy())
    else:
        turno = np.random.default_rng(42).permutation(len(vut))

    libre = hoteles["plazas"].to_numpy().astype(float).copy()
    filas = []
    for i in turno:
        pendientes = float(vut["accommodates"].iloc[i])
        for j in orden_hoteles[i]:
            if pendientes <= 0:
                break
            if libre[j] <= 0:
                continue
            cabe = min(pendientes, libre[j])
            libre[j] -= cabe
            pendientes -= cabe
            filas.append({
                "vut_idx": i, "hotel_idx": j, "plazas": cabe,
                "km": D[i, j],
                "salto_precio": hoteles["precio"].iloc[j] - vut["precio"].iloc[i],
                "partida": pendientes > 0,
            })
        if pendientes > 0:                                  # no deberia pasar: sobra capacidad
            filas.append({"vut_idx": i, "hotel_idx": -1, "plazas": pendientes,
                          "km": np.nan, "salto_precio": np.nan, "partida": True})
    return pd.DataFrame(filas), libre


asig, libre = repartir_con_capacidad(0.5)

print("=== REPARTO CON CAPACIDAD, w = 0,5 ===")
print(f"  plazas colocadas      : {asig.loc[asig.hotel_idx >= 0, 'plazas'].sum():>9,.0f}"
      f" de {vut['accommodates'].sum():,.0f}")
print(f"  sin sitio             : {asig.loc[asig.hotel_idx < 0, 'plazas'].sum():>9,.0f}")
print(f"  VUT partidas en varios: {asig[asig.hotel_idx >= 0].vut_idx.duplicated().sum():>9,}")
print(f"  hoteles usados        : {asig.loc[asig.hotel_idx >= 0, 'hotel_idx'].nunique():>9,}"
      f" de {len(hoteles):,}")
print(f"  hoteles llenos        : {int((libre <= 0).sum()):>9,}")
print(f"  ocupacion global      : {1 - libre.sum() / hoteles['plazas'].sum():>9.1%}")


=== REPARTO CON CAPACIDAD, w = 0,5 ===
  plazas colocadas      :    30,067 de 30,067
  sin sitio             :         0
  VUT partidas en varios:       312
  hoteles usados        :       663 de 762
  hoteles llenos        :       416
  ocupacion global      :     35.7%


In [11]:
# 11. Que cambia al meter la capacidad
comparativa = []
for w in [0.0, 0.5, 1.0]:
    libre_sin = None
    sin_cap = repartir(w)
    con_cap, _ = repartir_con_capacidad(w)
    ok = con_cap[con_cap.hotel_idx >= 0]
    comparativa.append({
        "w": w,
        "km medianos SIN capacidad": round(float(sin_cap["km_recorridos"].median()), 2),
        "km medianos CON capacidad": round(float(np.median(np.repeat(ok["km"], 1))), 2),
        "sobrecoste SIN": round(float(sin_cap["salto_precio"].median()), 1),
        "sobrecoste CON": round(float(ok["salto_precio"].median()), 1),
    })
display(pd.DataFrame(comparativa).set_index("w"))

print("La capacidad empuja al turista mas lejos y le cambia el precio: el hotel de al lado")
print("se llena con los primeros que llegan.")


,km medianos SIN capacidad,km medianos CON capacidad,sobrecoste SIN,sobrecoste CON
w,,,,
0.0,2.22,2.39,0.0,0.1
0.5,0.21,0.25,4.0,12.2
1.0,0.10,0.15,25.6,29.0


La capacidad empuja al turista mas lejos y le cambia el precio: el hotel de al lado
se llena con los primeros que llegan.


In [12]:
# 12. Sale sola la segmentacion por categoria?
#
# Si el modelo es razonable, las VUT caras deberian acabar en hoteles de mas estrellas sin que
# nadie lo haya impuesto.
asig, _ = repartir_con_capacidad(0.5)
ok = asig[asig.hotel_idx >= 0].copy()
ok["banda_vut"] = vut["banda_plaza"].to_numpy()[ok.vut_idx.to_numpy()]
ok["categoria_hotel"] = hoteles["tipo_alojamiento"].to_numpy()[ok.hotel_idx.to_numpy()]
estrellas = hoteles["estrellas"].to_numpy()[ok.hotel_idx.to_numpy()]
ok["destino"] = np.where(pd.isna(estrellas), ok["categoria_hotel"],
                         pd.Series(estrellas).fillna(0).astype(float).round(0).astype(int).astype(str) + " estrellas")

tabla = pd.crosstab(ok["banda_vut"], ok["destino"], values=ok["plazas"], aggfunc="sum")
tabla = tabla.reindex(["€", "€€", "€€€", "€€€€"]).fillna(0).astype(int)
print("=== PLAZAS: banda de la VUT (filas) -> categoria del hotel (columnas) ===")
display(tabla)

print("En porcentaje de cada banda:")
display((tabla.div(tabla.sum(axis=1), axis=0) * 100).round(0).fillna(0).astype(int))

media = ok.groupby("banda_vut").apply(
    lambda g: np.average(pd.Series(hoteles["precio"].to_numpy()[g.hotel_idx.to_numpy()]),
                         weights=g["plazas"]), include_groups=False).round(1)
print("Precio medio del hotel de destino, por banda de origen:")
print(media.reindex(["€", "€€", "€€€", "€€€€"]).to_string())


=== PLAZAS: banda de la VUT (filas) -> categoria del hotel (columnas) ===


destino,1 estrellas,2 estrellas,3 estrellas,4 estrellas,5 estrellas,apartament_turistic,sin_estrellas
banda_vut,,,,,,,
€,503,629,2661,4663,0,39,501
€€,1909,1764,4593,3797,0,440,3278
€€€,252,359,1090,1710,56,63,1294
€€€€,0,14,30,207,179,0,36


En porcentaje de cada banda:


destino,1 estrellas,2 estrellas,3 estrellas,4 estrellas,5 estrellas,apartament_turistic,sin_estrellas
banda_vut,,,,,,,
€,6,7,30,52,0,0,6
€€,12,11,29,24,0,3,21
€€€,5,7,23,35,1,1,27
€€€€,0,3,6,44,38,0,8


Precio medio del hotel de destino, por banda de origen:
banda_vut
€        76.8
€€       67.7
€€€      84.1
€€€€    148.6


In [13]:
# 13. Cuanto pesa elegir primero
#
# El turno por precio no es neutral. Comparado con un sorteo, ensena cuanto gana quien paga mas.
por_precio, _ = repartir_con_capacidad(0.5, prioridad="precio")
por_sorteo, _ = repartir_con_capacidad(0.5, prioridad="sorteo")

filas = []
for etiqueta, a in [("las caras eligen primero", por_precio), ("sorteo", por_sorteo)]:
    ok = a[a.hotel_idx >= 0].copy()
    ok["banda"] = vut["banda_plaza"].to_numpy()[ok.vut_idx.to_numpy()]
    for banda in ["€", "€€€€"]:
        g = ok[ok["banda"] == banda]
        if len(g):
            filas.append({"reparto": etiqueta, "banda VUT": banda,
                          "km medianos": round(float(g["km"].median()), 2),
                          "sobrecoste mediano": round(float(g["salto_precio"].median()), 1)})
display(pd.DataFrame(filas).set_index(["reparto", "banda VUT"]))


km medianos  sobrecoste mediano
reparto                  banda VUT                                 
las caras eligen primero €                 0.34                45.1
                         €€€€              0.37                -3.8
sorteo                   €                 0.29                37.5
                         €€€€              0.37                -3.7

In [14]:
# 14. Donde se llena la ciudad
asig, libre = repartir_con_capacidad(0.5)
ok = asig[asig.hotel_idx >= 0].copy()
ok["barrio"] = hoteles["barrio"].to_numpy()[ok.hotel_idx.to_numpy()]
ok["barrio_origen"] = vut["neighbourhood"].to_numpy()[ok.vut_idx.to_numpy()]

recibe = ok.groupby("barrio")["plazas"].sum()
capacidad = hoteles.groupby("barrio")["plazas"].sum()
barrios = pd.DataFrame({"recibidas": recibe, "capacidad": capacidad}).fillna(0)
barrios["ocupacion_%"] = (barrios["recibidas"] / barrios["capacidad"] * 100).round(0)

print("=== BARRIOS QUE SE LLENAN ===")
display(barrios.sort_values("ocupacion_%", ascending=False).head(10))

se_queda = (ok["barrio"] == ok["barrio_origen"])
print(f"plazas que se quedan en su propio barrio: "
      f"{ok.loc[se_queda, 'plazas'].sum():,.0f} de {ok['plazas'].sum():,.0f} "
      f"({ok.loc[se_queda, 'plazas'].sum() / ok['plazas'].sum():.0%})")


=== BARRIOS QUE SE LLENAN ===


,recibidas,capacidad,ocupacion_%
barrio,,,
Sant Genís dels Agudells,14.0,14.0,100.0
Sant Andreu,19.0,19.0,100.0
Navas,292.0,292.0,100.0
Vallcarca i els Penitents,112.0,112.0,100.0
Vilapicina i la Torre Llobeta,20.0,20.0,100.0
Sants - Badal,151.0,151.0,100.0
el Coll,254.0,254.0,100.0
la Bordeta,39.0,39.0,100.0
la Salut,58.0,58.0,100.0


plazas que se quedan en su propio barrio: 17,706 de 30,067 (59%)


# Que decidir

1. **Cual de las tres va a la web.** La 3 es la unica que da un numero de partida y deja
   discutirlo; la 1 no afirma nada; la 2 afirma mas de lo que el dato aguanta.
2. **Si el reparto respeta la capacidad hotelera.** Ahora no: un hotel puede recibir mas plazas de
   las que tiene. La tabla de la celda 8 ensena donde revienta.

El aviso que debe ir en la web pase lo que pase: **precio y distancia estan correlacionados** --el
centro es caro porque es el centro--, asi que repartir el peso entre los dos es una cuenta
discutible y no un hecho medido. La celda 5 lo cuantifica.